# 🔧 Predictive Maintenance — Machine Failure Forecasting

This notebook builds a **predictive maintenance model** that forecasts device failures before they occur.

---

## 📋 Table of Contents
1. [Imports & Setup](#1-imports--setup)
2. [Load Data](#2-load-data)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Feature Engineering](#4-feature-engineering)
5. [Train / Test Split](#5-train--test-split)
6. [Model Training (LightGBM)](#6-model-training-lightgbm)
7. [Model Evaluation](#7-model-evaluation)
8. [Probability Calibration](#8-probability-calibration)
9. [Top-K Risk Analysis](#9-top-k-risk-analysis)

---
## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    average_precision_score
)
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier

---
## 2. Load Data

In [2]:
file_path = "/kaggle/input/datasets/hiimanshuagarwal/predictive-maintenance-dataset/predictive_maintenance_dataset.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

Shape: (124494, 12)


,date,device,failure,metric1,metric2,metric3,metric4,metric5,metric6,metric7,metric8,metric9
0,1/1/2015,S1F01085,0,215630672,55,0,52,6,407438,0,0,7
1,1/1/2015,S1F0166B,0,61370680,0,3,0,6,403174,0,0,0
2,1/1/2015,S1F01E6Y,0,173295968,0,0,0,12,237394,0,0,0
3,1/1/2015,S1F01JE0,0,79694024,0,0,0,6,410186,0,0,0
4,1/1/2015,S1F01R2B,0,135970480,0,0,0,15,313173,0,0,3


In [3]:
# Column overview
print("Columns:\n", df.columns.tolist())

Columns:
 ['date', 'device', 'failure', 'metric1', 'metric2', 'metric3', 'metric4', 'metric5', 'metric6', 'metric7', 'metric8', 'metric9']


---
## 3. Exploratory Data Analysis

In [4]:
# Parse dates and sort by device + date
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['device', 'date']).reset_index(drop=True)

print("Date range:", df['date'].min(), "→", df['date'].max())
print("\nFailure rate:", df['failure'].mean().round(4))
print("\nFailure counts:")
print(df['failure'].value_counts())

Date range: 2015-01-01 00:00:00 → 2015-11-02 00:00:00

Failure rate: 0.0009

Failure counts:
failure
0    124388
1       106
Name: count, dtype: int64


---
## 4. Feature Engineering

We create the following features for each `metric1` through `metric9`:
- **Lag features** — previous day's value
- **Diff features** — day-over-day change
- **Rolling mean & std** — 7-day window statistics
- **Z-score** — device-level standardization

We also create a **future failure target label**: `target = 1` if a failure occurs within the next **45 days** for that device.

In [5]:
# --- Target: will this device fail within the next N days? ---
horizon_days = 45

def make_target(g):
    g = g.sort_values('date').copy()
    failure_dates = g.loc[g['failure'] == 1, 'date'].to_numpy()
    dates = g['date'].to_numpy()

    if len(failure_dates) == 0:
        g['target'] = 0
        return g

    right = np.searchsorted(
        failure_dates,
        dates + np.timedelta64(horizon_days, 'D'),
        side='right'
    )
    left = np.searchsorted(failure_dates, dates, side='right')
    g['target'] = (right > left).astype(int)
    return g

df = df.groupby('device', group_keys=False).apply(make_target)
print("Target distribution:")
print(df['target'].value_counts())

Target distribution:
target
0    120745
1      3749
Name: count, dtype: int64


/tmp/ipykernel_55/1212384509.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('device', group_keys=False).apply(make_target)


In [6]:
# --- Lag, diff, rolling, and z-score features per metric ---
metric_cols = [f'metric{i}' for i in range(1, 10)]

for col in metric_cols:
    g = df.groupby('device')[col]
    df[f'{col}_lag1']   = g.shift(1)
    df[f'{col}_diff1']  = g.diff()
    df[f'{col}_mean_7'] = g.rolling(7).mean().reset_index(0, drop=True)
    df[f'{col}_std_7']  = g.rolling(7).std().reset_index(0, drop=True)
    df[f'{col}_z']      = (df[col] - g.transform('mean')) / (g.transform('std') + 1e-6)

# Drop rows with NaNs introduced by rolling/lag
df = df.dropna().reset_index(drop=True)
print("Shape after feature engineering:", df.shape)

Shape after feature engineering: (117596, 58)


---
## 5. Train / Test Split

We split by **time** to avoid data leakage — all data before `2015-07-15` is training, the rest is test.

In [7]:
split_date = pd.Timestamp('2015-07-15')
train = df[df['date'] < split_date].copy()
test  = df[df['date'] >= split_date].copy()

print(f"Train: {len(train):,} rows | Test: {len(test):,} rows")
print(f"Train failure rate: {train['failure'].mean():.4f}  |  Test failure rate: {test['failure'].mean():.4f}")

Train: 96,100 rows | Test: 21,496 rows
Train failure rate: 0.0009  |  Test failure rate: 0.0007


In [8]:
# Define feature columns (all metric raw + engineered, excluding identifiers/labels)
exclude = {'date', 'device', 'failure', 'target'}
features = [c for c in df.columns if c not in exclude]

X_train = train[features]
y_train = train['target']
X_test  = test[features]
y_test  = test['target']

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)
print("\nTarget class balance (train):")
print(y_train.value_counts())
print("\nTarget class balance (test):")
print(y_test.value_counts())

X_train shape: (96100, 54)
X_test  shape: (21496, 54)

Target class balance (train):
target
0    92847
1     3253
Name: count, dtype: int64

Target class balance (test):
target
0    21201
1      295
Name: count, dtype: int64


---
## 6. Model Training (LightGBM)

We use **LightGBM** with inverse class-frequency weighting to handle the class imbalance.

In [9]:
# Class weight to handle imbalance
scale = (len(y_train) - y_train.sum()) / y_train.sum()

model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    class_weight={0: 1, 1: scale}
)

model.fit(X_train, y_train)
print("✅ Model training complete.")

[LightGBM] [Info] Number of positive: 3253, number of negative: 92847
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020727 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6340
[LightGBM] [Info] Number of data points in the train set: 96100, number of used features: 53
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
✅ Model training complete.


---
## 7. Model Evaluation

We evaluate using:
- **Precision / Recall** at multiple thresholds
- **Classification report** at the 98th percentile cutoff
- **PR-AUC** (preferred over ROC-AUC for imbalanced datasets)

In [10]:
# Get predicted probabilities from base model
y_prob = model.predict_proba(X_test)[:, 1]

print("Percentile check (90th / 95th / 99th):")
print(np.percentile(y_prob, [90, 95, 99]))

Percentile check (90th / 95th / 99th):
[0.59889991 0.8747848  0.98435241]


In [11]:
# Precision / Recall at multiple thresholds
print(f"{'Threshold':<12} {'Recall':<10} {'Precision'}")
print("-" * 35)
for t in [0.01, 0.02, 0.05, 0.1]:
    y_pred = (y_prob > t).astype(int)
    r = recall_score(y_test, y_pred)
    p = precision_score(y_test, y_pred)
    print(f"{t:<12} {r:<10.4f} {p:.4f}")

Threshold    Recall     Precision
-----------------------------------
0.01         0.9627     0.0224
0.02         0.9390     0.0265
0.05         0.8915     0.0354
0.1          0.8576     0.0461


In [12]:
# Classification report at top-2% threshold
threshold_98 = np.percentile(y_prob, 98)
y_pred_98 = (y_prob >= threshold_98).astype(int)

print("Classification Report (Top 2% threshold)\n")
print(classification_report(y_test, y_pred_98))

Classification Report (Top 2% threshold)

              precision    recall  f1-score   support

           0       0.99      0.98      0.99     21201
           1       0.13      0.20      0.16       295

    accuracy                           0.97     21496
   macro avg       0.56      0.59      0.57     21496
weighted avg       0.98      0.97      0.97     21496



In [13]:
# Top-K failure catch rate (base model)
print("Top-K Failure Catch Rate (Base Model)")
print(f"{'Top %':<10} {'Caught':<10} {'Total Failures'}")
print("-" * 35)
for p in [0.01, 0.02, 0.05, 0.90]:
    k = int(p * len(y_prob))
    idx = np.argsort(-y_prob)[:k]
    caught = y_test.iloc[idx].sum()
    print(f"Top {int(p*100):>3}%    {int(caught):<10} {int(y_test.sum())}")

Top-K Failure Catch Rate (Base Model)
Top %      Caught     Total Failures
-----------------------------------
Top   1%    27         295
Top   2%    57         295
Top   5%    132        295
Top  90%    295        295


---
## 8. Probability Calibration

LightGBM probabilities can be poorly calibrated (overconfident or underconfident). We apply **isotonic regression** calibration to improve probability estimates.

In [14]:
# Calibrate probabilities using isotonic regression
cal_model = CalibratedClassifierCV(model, method='isotonic')
cal_model.fit(X_train, y_train)

y_prob_cal = cal_model.predict_proba(X_test)[:, 1]
print("✅ Calibration complete.")

[LightGBM] [Info] Number of positive: 2603, number of negative: 74277
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015815 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6039
[LightGBM] [Info] Number of data points in the train set: 76880, number of used features: 53
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500060 -> initscore=0.000239
[LightGBM] [Info] Start training from score 0.000239
[LightGBM] [Info] Number of positive: 2603, number of negative: 74277
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015515 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6150
[LightGBM] [Info] Number of data points in the train set: 76880, number of used features: 53
[LightGBM] [Info] [bin

In [15]:
# Compare original vs calibrated probability distributions
print("Percentile comparison (90th / 95th / 99th):")
print("Original   :", np.percentile(y_prob,     [90, 95, 99]))
print("Calibrated :", np.percentile(y_prob_cal, [90, 95, 99]))

Percentile comparison (90th / 95th / 99th):
Original   : [0.59889991 0.8747848  0.98435241]
Calibrated : [0.34079887 0.43176405 0.5762006 ]


In [16]:
# Precision / Recall at multiple thresholds (calibrated)
print(f"{'Threshold':<12} {'Recall':<10} {'Precision'}")
print("-" * 35)
for t in [0.01, 0.02, 0.05, 0.1]:
    y_pred = (y_prob_cal > t).astype(int)
    r = recall_score(y_test, y_pred)
    p = precision_score(y_test, y_pred)
    print(f"{t:<12} {r:<10.4f} {p:.4f}")

Threshold    Recall     Precision
-----------------------------------
0.01         1.0000     0.0159
0.02         1.0000     0.0179
0.05         0.9966     0.0209
0.1          0.9831     0.0319


In [17]:
# PR-AUC comparison: original vs calibrated
pr_auc_orig = average_precision_score(y_test, y_prob)
pr_auc_cal  = average_precision_score(y_test, y_prob_cal)

print(f"PR-AUC (Original)  : {pr_auc_orig:.4f}")
print(f"PR-AUC (Calibrated): {pr_auc_cal:.4f}")

PR-AUC (Original)  : 0.1300
PR-AUC (Calibrated): 0.1452


---
## 9. Top-K Risk Analysis

In predictive maintenance, we care most about: **out of the riskiest X% of devices, how many actual failures do we catch?**

In [18]:
# Top-K failure catch rate (calibrated model)
print("Top-K Failure Catch Rate (Calibrated Model)")
print(f"{'Top %':<10} {'Caught':<10} {'Total Failures'}")
print("-" * 35)
for p in [0.01, 0.02, 0.05]:
    k = int(p * len(y_prob_cal))
    idx = np.argsort(-y_prob_cal)[:k]
    caught = y_test.iloc[idx].sum()
    print(f"Top {int(p*100):>3}%    {int(caught):<10} {int(y_test.sum())}")

Top-K Failure Catch Rate (Calibrated Model)
Top %      Caught     Total Failures
-----------------------------------
Top   1%    45         295
Top   2%    87         295
Top   5%    140        295


In [19]:
# Final predictions (binary) using calibrated model
y_final_pred = cal_model.predict(X_test)
print("Sample predictions (first 10):", y_final_pred[:10])

Sample predictions (first 10): [0 0 0 0 0 0 0 0 0 0]
